# Nozzle inverse by native QSVT

Fit fresh phase sequences and solve both original nozzle systems with FlagQuantum.

Both complete inverse streams now use native qfunc composition: source-state
preparation, selected blocks, adjoints and projector phases. The host derives
classical tables and fits the original 102/256 phases; synth allocates and cleans
scratch and lowers these operations to gates. No packed response matrix enters
the executed program. The separate Circuit builder remains an independent reference.

Inspect complete complex-state error, clean work probability, reciprocal response
and selector-zero solution rescaling separately. The original matrices, right-hand
sides and all non-success branches remain part of acceptance. These bounded CPU
examples do not claim hardware speedup or a general linear-solver API.


Use the local FlagQuantum environment.

In [ ]:
from pathlib import Path

import torch
from IPython.display import Image, display

import flagquantum as fq
from examples.synthesis.applications.cfd.qls_for_hybrid_solvers.qls_qsvt.overview import (
    native,
)
from examples.synthesis.applications.cfd.qls_for_hybrid_solvers.qls_qsvt.overview.native import (
    band_term,
    pauli_signal,
)
from examples.synthesis.applications.cfd.qls_for_hybrid_solvers.qls_qsvt.overview.workflow import (
    CTYPE,
)
from flagquantum import synth
from flagquantum.synth import (
    H,
    Output,
    QArray,
    QBit,
    QNum,
    adjoint,
    allocate,
    apply,
    control,
    inplace_prepare_amplitudes,
    phase,
    prepare_amplitudes,
    qfunc,
    within,
)


@qfunc
def pauli_inverse(
    amplitudes: list[float],
    weights: list[float],
    codes: list[float],
    angles: list[float],
    count: int,
    phases: list[float],
    pairs: int,
    branch: Output[QBit],
    selection: Output[QNum[5]],
    data: Output[QArray[QBit]],
) -> None:
    allocate(branch)
    allocate(selection)
    prepare_amplitudes(amplitudes, 0, data)
    H(branch)
    phase(1, -(phases[0]))
    phase(branch, 2 * (phases[0]))
    with within():
        success: QNum[1] = selection == 0
    with apply():
        phase(success, 2 * (phases[0]))
        control(success, lambda: phase(branch, -4 * (phases[0])))
    for pair in range(pairs):
        with adjoint():
            pauli_signal(weights, codes, angles, count, selection, data)
        phase(1, -(phases[2 * pair + 1]))
        phase(branch, 2 * (phases[2 * pair + 1]))
        with within():
            success: QNum[1] = selection == 0
        with apply():
            phase(success, 2 * (phases[2 * pair + 1]))
            control(
                success,
                lambda: phase(branch, -4 * (phases[2 * pair + 1])),  # noqa: B023
            )
        pauli_signal(weights, codes, angles, count, selection, data)
        phase(1, -(phases[2 * pair + 2]))
        phase(branch, 2 * (phases[2 * pair + 2]))
        with within():
            success: QNum[1] = selection == 0
        with apply():
            phase(success, 2 * (phases[2 * pair + 2]))
            control(
                success,
                lambda: phase(branch, -4 * (phases[2 * pair + 2])),  # noqa: B023
            )
    with adjoint():
        pauli_signal(weights, codes, angles, count, selection, data)
    phase(1, -(phases[2 * pairs + 1]))
    phase(branch, 2 * (phases[2 * pairs + 1]))
    with within():
        success: QNum[1] = selection == 0
    with apply():
        phase(success, 2 * (phases[2 * pairs + 1]))
        control(success, lambda: phase(branch, -4 * (phases[2 * pairs + 1])))
    H(branch)



Import the native QSVT solve.

In [ ]:
from examples.synthesis.applications.cfd.qls_for_hybrid_solvers.chebyshev_approximation.overview.model import (
    source_matrix,
)
from examples.synthesis.applications.cfd.qls_for_hybrid_solvers.qls_chebyshev_lcu.overview.workflow import (
    input_vector,
)
from examples.synthesis.applications.cfd.qls_for_hybrid_solvers.qls_qsvt.overview.main import (
    print_report,
    save_figure,
)
from examples.synthesis.applications.cfd.qls_for_hybrid_solvers.qls_qsvt.overview.workflow import (
    solve,
)
from examples.synthesis.applications.cfd.qls_for_hybrid_solvers.verify_block_encoding.overview.model import (
    source_matrix as small_matrix,
)

print(fq.__name__)


Fit each odd inverse polynomial and apply its alternating block and adjoint sequence.

In [ ]:
print("Original inverse polynomial degrees:", 101, 255)


Select the CPU simulator for the original-size examples.

In [ ]:
print("Execution target: local CPU")


Choose the output figure directory.

In [ ]:
figures = Path("notebook-figures")


Read the original eight-row system and right-hand side.

In [ ]:
matrix_small = small_matrix()
rhs_small = input_vector("b_nozzle_small", 3)
print(matrix_small.shape, rhs_small.shape)


Regenerate 102 phases and execute the degree-101 Pauli solve.

In [ ]:
small = solve(False)
print_report("Small", small)


Compare the postselected small solution with direct linear algebra.

In [ ]:
save_figure(small, "Small nozzle inverse", figures / "nozzle-small.png")
display(Image(filename=str(figures / "nozzle-small.png")))


Check the original small-system error limit.

In [ ]:
assert small.solution_l2 < 0.1
assert small.workspace_probability < 1e-24


Read the original sixteen-row system and right-hand side.

In [ ]:
matrix_large = source_matrix()
rhs_large = input_vector("nozzle_008_b", 4)
print(matrix_large.shape, rhs_large.shape)


## Compose the original six-band block and full inverse

The sixteen-row matrix occupies offsets -8, -7, -1, 0, 7 and 8, retaining
all 45 entries. The borrowed signal uses bounded rotation tables and modular
shifts, with selector preparation and its inverse supplied by within/apply.
The inverse alternates this signal and its adjoint across every original phase.
Each projector computes one shared success predicate, uses it coherently and
uncomputes it. No manual work-bit allocation or wire map is needed.

The source block uses eleven total qubits and three work bits. The complete
256-phase inverse uses twelve total qubits, three work bits and 87012 logical
instructions under the unchanged 100000-instruction construction ceiling.
The small 102-phase inverse uses thirteen total qubits and four work bits.


In [ ]:
@qfunc
def banded_signal(
    weights: list[float],
    angles: list[float],
    offsets: list[int],
    count: int,
    selection: QNum[3],
    flag: QBit,
    data: QNum[4],
) -> None:
    with within():
        inplace_prepare_amplitudes(weights, 0, selection)
    with apply():
        for term in range(count):
            control(
                selection == term,
                lambda: band_term(angles, offsets, term, flag, data),  # noqa: B023
            )


@qfunc
def banded_block(
    amplitudes: list[float],
    weights: list[float],
    angles: list[float],
    offsets: list[int],
    count: int,
    selection: Output[QNum[3]],
    flag: Output[QBit],
    data: Output[QNum[4]],
) -> None:
    allocate(selection)
    allocate(flag)
    prepare_amplitudes(amplitudes, 0, data)
    banded_signal(weights, angles, offsets, count, selection, flag, data)


@qfunc
def banded_inverse(
    amplitudes: list[float],
    weights: list[float],
    angles: list[float],
    offsets: list[int],
    count: int,
    phases: list[float],
    pairs: int,
    branch: Output[QBit],
    selection: Output[QNum[3]],
    flag: Output[QBit],
    data: Output[QNum[4]],
) -> None:
    allocate(branch)
    allocate(selection)
    allocate(flag)
    prepare_amplitudes(amplitudes, 0, data)
    H(branch)
    phase(1, -(phases[0]))
    phase(branch, 2 * (phases[0]))
    with within():
        success: QNum[1] = (selection == 0) & (flag == 0)
    with apply():
        phase(success, 2 * (phases[0]))
        control(success, lambda: phase(branch, -4 * (phases[0])))
    for pair in range(pairs):
        with adjoint():
            banded_signal(weights, angles, offsets, count, selection, flag, data)
        phase(1, -(phases[2 * pair + 1]))
        phase(branch, 2 * (phases[2 * pair + 1]))
        with within():
            success: QNum[1] = (selection == 0) & (flag == 0)
        with apply():
            phase(success, 2 * (phases[2 * pair + 1]))
            control(
                success,
                lambda: phase(branch, -4 * (phases[2 * pair + 1])),  # noqa: B023
            )
        banded_signal(weights, angles, offsets, count, selection, flag, data)
        phase(1, -(phases[2 * pair + 2]))
        phase(branch, 2 * (phases[2 * pair + 2]))
        with within():
            success: QNum[1] = (selection == 0) & (flag == 0)
        with apply():
            phase(success, 2 * (phases[2 * pair + 2]))
            control(
                success,
                lambda: phase(branch, -4 * (phases[2 * pair + 2])),  # noqa: B023
            )
    with adjoint():
        banded_signal(weights, angles, offsets, count, selection, flag, data)
    phase(1, -(phases[2 * pairs + 1]))
    phase(branch, 2 * (phases[2 * pairs + 1]))
    with within():
        success: QNum[1] = (selection == 0) & (flag == 0)
    with apply():
        phase(success, 2 * (phases[2 * pairs + 1]))
        control(success, lambda: phase(branch, -4 * (phases[2 * pairs + 1])))
    H(branch)


# The host builder only derives diagonal offsets, scales and real angle tables.
bindings, block_scale = native.banded_inputs(matrix_large, rhs_large / rhs_large.norm())
block_program = synth.synthesize(
    banded_block,
    classical_inputs=bindings,
    max_qubits=11,
    max_workspace=3,
    max_operations=1024,
    memory_limit_bytes=64 * 1024**2,
)
block_result = fq.run(block_program)
block_state = block_result.statevector().reshape(256, 8)
torch.testing.assert_close(
    block_state[:16, 0],
    matrix_large.to(CTYPE) @ (rhs_large / rhs_large.norm()).to(CTYPE) / block_scale,
    atol=1e-12,
    rtol=0,
)
assert torch.count_nonzero(block_state[:, 1:]) == 0
print("Native banded block:", block_program.report["resources"])
print("Clean work bits:", block_program.report["workspace_qubits"])
print(
    "Block success:",
    block_program.probabilities(
        block_result,
        registers=("selection", "flag"),
    )[(0, 0)],
)


Regenerate 256 phases and execute the degree-255 banded solve.

In [ ]:
large = solve(True)
print_report("Large", large)


Compare the postselected large solution with the dense reference.

In [ ]:
save_figure(large, "Large nozzle inverse", figures / "nozzle-large.png")
display(Image(filename=str(figures / "nozzle-large.png")))


Check the original large-system error limit.

In [ ]:
assert large.solution_l2 < 0.1
assert large.workspace_probability < 1e-24
